# 01 — Análise Exploratória de Dados (EDA)
**Dataset:** KC House Sales (King County, WA — 2014/2015)  
**Objetivo:** Entender distribuições, correlações, sazonalidade e padrões geográficos antes de modelar.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

# Tema global
import plotly.io as pio
pio.templates.default = 'plotly_white'

BLUE   = '#2563EB'
RED    = '#DC2626'
GRAY   = '#6B7280'
GREEN  = '#16A34A'
ORANGE = '#EA580C'

DATA_PATH = Path('../data/raw/kc_house_data.csv')
print('Libs ok.')

## 1. Carga e limpeza básica

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f'Shape bruto: {df_raw.shape}')
df_raw.head(3)

In [ ]:
# Mesma limpeza de app/ml/preprocess.py
df = df_raw.sort_values('date', ascending=False).drop_duplicates(subset='id')
df = df[df['price'] > 0]
df = df[df['bedrooms'] <= 20]
df['zipcode'] = df['zipcode'].astype(str)
df['yr_renovated'] = df['yr_renovated'].fillna(0).astype(int)
df['date_dt'] = pd.to_datetime(df['date'].astype(str).str[:8], format='%Y%m%d')
df['sale_month'] = df['date_dt'].dt.month
df['sale_year']  = df['date_dt'].dt.year
df['month_label'] = df['date_dt'].dt.to_period('M').astype(str)

REFERENCE_YEAR = 2015
df['house_age']          = REFERENCE_YEAR - df['yr_built']
df['was_renovated']      = (df['yr_renovated'] > 0).astype(int)
df['living_lot_ratio']   = df['sqft_living'] / (df['sqft_lot'] + 1)
df['bath_bed_ratio']     = df['bathrooms'] / (df['bedrooms'] + 1)
df['has_basement']       = (df['sqft_basement'] > 0).astype(int)
df['price_k']            = df['price'] / 1_000
df['log_price']          = np.log1p(df['price'])

print(f'Shape limpo : {df.shape}')
print(f'Período     : {df["date_dt"].min().date()} → {df["date_dt"].max().date()}')
print(f'Zipcodes    : {df["zipcode"].nunique()}')

## 2. Distribuição do alvo — `price`

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Escala original', 'Escala log1p — usada no modelo']
)

fig.add_trace(
    go.Histogram(x=df['price_k'], nbinsx=80, marker_color=BLUE,
                 marker_line_width=0.3, marker_line_color='white',
                 name='price'),
    row=1, col=1
)
fig.add_trace(
    go.Histogram(x=df['log_price'], nbinsx=80, marker_color=ORANGE,
                 marker_line_width=0.3, marker_line_color='white',
                 name='log1p(price)'),
    row=1, col=2
)

fig.update_xaxes(title_text='Preço (k$)', row=1, col=1)
fig.update_xaxes(title_text='log1p(price)', row=1, col=2)
fig.update_yaxes(title_text='Frequência', row=1, col=1)

fig.update_layout(
    title_text='Distribuição do preço de venda',
    title_font_size=16,
    showlegend=False,
    height=420,
)
fig.show()

stats = df['price'].describe()
print(stats.apply(lambda x: f'${x:,.0f}').to_string())

## 3. Sazonalidade — volume e preço mediano por mês

In [ ]:
monthly = (
    df.groupby('month_label')
    .agg(n_vendas=('price', 'count'), preco_mediano=('price', 'median'))
    .reset_index()
    .sort_values('month_label')
)

fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(
    go.Bar(x=monthly['month_label'], y=monthly['n_vendas'],
           name='Nº de vendas', marker_color=BLUE, opacity=0.6),
    secondary_y=False
)
fig.add_trace(
    go.Scatter(x=monthly['month_label'], y=monthly['preco_mediano'] / 1_000,
               name='Preço mediano (k$)', mode='lines+markers',
               line=dict(color=RED, width=2.5), marker=dict(size=6)),
    secondary_y=True
)

fig.update_layout(
    title_text='Sazonalidade: volume de vendas e preço mediano',
    title_font_size=16,
    xaxis_title='Mês',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=430,
)
fig.update_yaxes(title_text='Nº de vendas', secondary_y=False)
fig.update_yaxes(title_text='Preço mediano (k$)', secondary_y=True)
fig.show()

In [ ]:
# Padrão sazonal agregado (independente do ano)
month_names = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
pattern = df.groupby('sale_month')['price_k'].median().reset_index()
pattern['mes'] = pattern['sale_month'].apply(lambda m: month_names[m-1])

fig = px.bar(
    pattern, x='mes', y='price_k',
    title='Padrão sazonal — preço mediano por mês do ano',
    labels={'price_k': 'Preço mediano (k$)', 'mes': 'Mês'},
    color='price_k',
    color_continuous_scale='Blues',
    text_auto='.0f',
)
fig.update_traces(textposition='outside', marker_line_width=0)
fig.update_layout(coloraxis_showscale=False, height=400, title_font_size=16)
fig.show()

## 4. Correlações com o preço

In [ ]:
numeric_cols = [
    'bedrooms','bathrooms','sqft_living','sqft_lot','floors',
    'waterfront','view','condition','grade','sqft_above',
    'sqft_basement','sqft_living15','sqft_lot15',
    'house_age','living_lot_ratio','bath_bed_ratio',
]

corr = df[numeric_cols + ['price']].corr()['price'].drop('price').sort_values()

colors = [RED if v < 0 else BLUE for v in corr.values]

fig = go.Figure(go.Bar(
    x=corr.values,
    y=corr.index,
    orientation='h',
    marker_color=colors,
    text=[f'{v:.3f}' for v in corr.values],
    textposition='outside',
))
fig.add_vline(x=0, line_width=1, line_color=GRAY)
fig.update_layout(
    title='Correlação de Pearson com price',
    title_font_size=16,
    xaxis_title='Correlação',
    height=550,
    margin=dict(l=160),
)
fig.show()

In [ ]:
# Heatmap das top features
top_features = corr.abs().nlargest(10).index.tolist() + ['price']
corr_matrix = df[top_features].corr().round(2)

fig = px.imshow(
    corr_matrix,
    text_auto=True,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Heatmap — top 10 features correlacionadas com price',
    aspect='auto',
)
fig.update_layout(title_font_size=16, height=520)
fig.show()

## 5. Features contínuas vs. preço

In [ ]:
scatter_features = [
    ('sqft_living', 'sqft_living (ft²)'),
    ('grade', 'grade'),
    ('bathrooms', 'bathrooms'),
    ('sqft_living15', 'sqft_living15 (ft²)'),
    ('view', 'view'),
    ('waterfront', 'waterfront'),
]

fig = make_subplots(rows=2, cols=3,
                    subplot_titles=[label for _, label in scatter_features])

positions = [(1,1),(1,2),(1,3),(2,1),(2,2),(2,3)]

for (feat, label), (row, col) in zip(scatter_features, positions):
    fig.add_trace(
        go.Scattergl(
            x=df[feat], y=df['price_k'],
            mode='markers',
            marker=dict(color=BLUE, opacity=0.15, size=3),
            showlegend=False,
            hovertemplate=f'{label}: %{{x}}<br>Price: $%{{y:.0f}}k<extra></extra>',
        ),
        row=row, col=col
    )
    fig.update_yaxes(title_text='Price (k$)', row=row, col=col)
    fig.update_xaxes(title_text=label, row=row, col=col)

fig.update_layout(
    title_text='Features contínuas vs. Preço',
    title_font_size=16,
    height=700,
)
fig.show()

## 6. Distribuição por `grade` e `condition`

In [ ]:
for col in ['grade', 'condition']:
    agg = df.groupby(col).agg(
        preco_mediano=('price_k', 'median'),
        n=('price', 'count')
    ).reset_index()

    fig = make_subplots(specs=[[{'secondary_y': True}]])

    fig.add_trace(
        go.Bar(x=agg[col].astype(str), y=agg['preco_mediano'],
               name='Preço mediano (k$)', marker_color=BLUE, opacity=0.7,
               text=agg['preco_mediano'].round(0).astype(int),
               textposition='outside'),
        secondary_y=False
    )
    fig.add_trace(
        go.Scatter(x=agg[col].astype(str), y=agg['n'],
                   name='Nº de imóveis', mode='lines+markers',
                   line=dict(color=RED, width=2), marker=dict(size=7)),
        secondary_y=True
    )

    fig.update_layout(
        title_text=f'Preço mediano e volume por {col}',
        title_font_size=15,
        xaxis_title=col,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        height=420,
    )
    fig.update_yaxes(title_text='Preço mediano (k$)', secondary_y=False)
    fig.update_yaxes(title_text='Nº de imóveis', secondary_y=True)
    fig.show()

## 7. Análise geográfica — mapa de preços por lat/long

In [ ]:
fig = px.scatter_mapbox(
    df.sample(5000, random_state=42),  # amostra para performance
    lat='lat', lon='long',
    color='price_k',
    color_continuous_scale='RdYlGn',
    size_max=6,
    zoom=9,
    mapbox_style='carto-positron',
    title='Distribuição geográfica de preços — King County',
    labels={'price_k': 'Preço (k$)'},
    hover_data={'price_k': ':.0f', 'zipcode': True, 'grade': True, 'sqft_living': True},
    opacity=0.65,
    height=600,
)
fig.update_layout(title_font_size=16, margin=dict(l=0, r=0, t=40, b=0))
fig.show()

In [ ]:
# Top / bottom zipcodes
zip_stats = (
    df.groupby('zipcode')
    .agg(preco_mediano=('price_k', 'median'), n=('price', 'count'))
    .reset_index()
    .sort_values('preco_mediano', ascending=False)
)

top10    = zip_stats.head(10).copy()
bottom10 = zip_stats.tail(10).copy()
combined = pd.concat([top10.assign(grupo='Top 10'), bottom10.assign(grupo='Bottom 10')])

fig = px.bar(
    combined,
    x='zipcode', y='preco_mediano', color='grupo',
    color_discrete_map={'Top 10': BLUE, 'Bottom 10': RED},
    text='preco_mediano',
    title='Top 10 e Bottom 10 zipcodes por preço mediano',
    labels={'preco_mediano': 'Preço mediano (k$)', 'zipcode': 'Zipcode'},
    height=430,
)
fig.update_traces(texttemplate='$%{text:.0f}k', textposition='outside')
fig.update_layout(title_font_size=16, legend_title_text='')
fig.show()

## 8. Split temporal — treino vs. teste

In [ ]:
CUTOFF = pd.Timestamp('2015-01-01')
df['split'] = df['date_dt'].apply(lambda d: 'Treino (< 2015-01-01)' if d < CUTOFF else 'Teste (≥ 2015-01-01)')

counts = df['split'].value_counts()
print(counts.to_string())
print(f"\nPreço médio treino : ${df[df['split'].str.startswith('Treino')]['price'].mean():,.0f}")
print(f"Preço médio teste  : ${df[df['split'].str.startswith('Teste')]['price'].mean():,.0f}")

In [ ]:
fig = px.histogram(
    df, x='price_k', color='split',
    nbins=70, barmode='overlay', opacity=0.65,
    color_discrete_map={
        'Treino (< 2015-01-01)': BLUE,
        'Teste (≥ 2015-01-01)': RED,
    },
    title='Distribuição de preços — treino vs. teste (split temporal 01/01/2015)',
    labels={'price_k': 'Preço (k$)', 'split': ''},
    histnorm='probability density',
    height=430,
)

for grupo, color in [('Treino (< 2015-01-01)', BLUE), ('Teste (≥ 2015-01-01)', RED)]:
    mediana = df[df['split'] == grupo]['price_k'].median()
    fig.add_vline(
        x=mediana, line_dash='dash', line_color=color, line_width=1.8,
        annotation_text=f'mediana {grupo[:6]}: ${mediana:.0f}k',
        annotation_position='top right',
    )

fig.update_layout(
    title_font_size=16,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

## 9. Outliers

In [ ]:
p99 = df['price'].quantile(0.99)
df['outlier'] = df['price'] > p99

fig = px.scatter(
    df, x='sqft_living', y='price_k',
    color='outlier',
    color_discrete_map={False: BLUE, True: RED},
    opacity=0.4,
    size_max=6,
    title=f'Outliers: imóveis acima do p99 (${p99:,.0f})',
    labels={'sqft_living': 'sqft_living', 'price_k': 'Preço (k$)', 'outlier': 'Outlier (p99)'},
    hover_data=['zipcode', 'grade', 'bedrooms'],
    height=450,
)
fig.add_hline(y=p99/1000, line_dash='dot', line_color=RED,
              annotation_text=f'p99 = ${p99/1000:.0f}k',
              annotation_position='top left')
fig.update_layout(title_font_size=16)
fig.show()

print(f'\nOutliers (>{p99:,.0f}): {df["outlier"].sum()} imóveis ({df["outlier"].mean()*100:.1f}%)')

## 10. Resumo estatístico

In [ ]:
summary_cols = ['price','sqft_living','bedrooms','bathrooms','grade','condition','house_age']
df[summary_cols].describe().round(2)

---
**Conclusões principais:**
- Preço fortemente skewed à direita → transformação log1p é correta.
- `sqft_living`, `grade`, `lat` e `zipcode` são os preditores mais fortes.
- Sazonalidade clara: pico de vendas e preços em Abr–Jun, queda em Dez–Jan.
- Split temporal 01/01/2015 divide ~80%/20% sem leakage temporal.
- Ver `notebooks/02_walk_forward_validation.ipynb` para validação multi-fold.